# LeetCode 860 — Lemonade Change bằng Python

**Link đề:** https://leetcode.com/problems/lemonade-change/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy

```python
class Solution:
    def lemonadeChange(self, bills: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Mô phỏng một quầy bán hàng: theo dõi **trạng thái** (số tờ tiền mỗi loại) qua từng khách.
2. Hiểu bài toán có **lựa chọn**: thối 15$ bằng `10 + 5` hay `5 + 5 + 5`? — và vì sao chọn **tham lam (greedy)** `10 + 5` luôn đúng.
3. Đi từ thử mọi lựa chọn (đệ quy / tập trạng thái — chậm) → greedy với hai biến đếm `O(n)`, `O(1)`.
4. Học cách **chứng minh greedy** bằng lập luận "trao đổi" (exchange argument): tờ 5$ "linh hoạt" hơn tờ 10$.
5. Trace, test biên (khách đầu trả 10/20, chỉ có tờ 10), và luyện các biến thể tiền thối.

## 1. Đề bài và ví dụ

Mỗi ly nước chanh giá **5$**. Khách xếp hàng mua lần lượt theo `bills`; mỗi người mua **một** ly và trả bằng tờ **5$, 10$ hoặc 20$**. Bạn phải thối lại đúng để mỗi khách thực trả 5$. Ban đầu bạn **không có** tiền lẻ. Trả `True` nếu thối được cho **mọi** khách.

- Khách trả 5$ → không cần thối.
- Khách trả 10$ → thối 5$.
- Khách trả 20$ → thối 15$: `10 + 5` hoặc `5 + 5 + 5`. (Tờ 20$ nhận vào **không bao giờ** dùng để thối được, vì không có món thối nào ≥ 20$.)

| bills | Kết quả | Giải thích |
|---|---|---|
| `[5,5,5,10,20]` | `True` | 3 khách đầu cho 3 tờ 5; khách 4 thối 5; khách 5 thối 10 + 5 |
| `[5,5,10,10,20]` | `False` | đến khách cuối chỉ còn hai tờ 10, không thối được 15 |

**Constraints:** `1 <= len(bills) <= 10^5`; `bills[i]` ∈ `{5, 10, 20}`.

In [1]:
bills = [5, 5, 5, 10, 20]
for i, bill in enumerate(bills):
    print(f"khách {i}: trả {bill}$ -> cần thối {bill - 5}$")

khách 0: trả 5$ -> cần thối 0$
khách 1: trả 5$ -> cần thối 0$
khách 2: trả 5$ -> cần thối 0$
khách 3: trả 10$ -> cần thối 5$
khách 4: trả 20$ -> cần thối 15$


## 2. Nền tảng Python và tư duy cần dùng

### 2.1. Biến đếm trạng thái

Ta chỉ cần biết **số tờ** mỗi loại đang có, không cần biết tờ nào của khách nào. Hai biến `five`, `ten` là đủ (tờ 20 không dùng để thối nên không cần đếm).

In [2]:
five, ten = 0, 0
five += 1          # nhận một tờ 5
five += 1
five -= 1; ten += 1  # khách trả 10: nhận tờ 10, thối một tờ 5
print("five =", five, "| ten =", ten)

five = 1 | ten = 1


### 2.2. `if / elif / else` với nhiều nhánh

Mỗi khách rơi vào **đúng một** nhánh theo mệnh giá. Thứ tự các `elif` quan trọng khi các điều kiện có thể cùng đúng (ở nhánh 20$ ta thử `10 + 5` **trước** `5 + 5 + 5`).

In [3]:
def classify(bill):
    if bill == 5:
        return "không thối"
    elif bill == 10:
        return "thối 5"
    else:
        return "thối 15"

print([classify(x) for x in [5, 10, 20]])

['không thối', 'thối 5', 'thối 15']


### 2.3. `dict` làm "ngăn kéo tiền"

Một cách tổng quát hơn là dùng `dict` mệnh giá → số tờ. `drawer.get(k, 0)` đọc an toàn khi khóa chưa có.

In [4]:
drawer = {5: 0, 10: 0, 20: 0}
drawer[5] += 2
drawer[10] += 1
print(drawer, drawer.get(50, 0))

{5: 2, 10: 1, 20: 0} 0


### 2.4. Tham lam (greedy) là gì?

**Greedy:** tại mỗi bước chọn phương án "tốt nhất trước mắt" theo một quy tắc cố định, **không quay lui**. Greedy chỉ đúng khi ta **chứng minh** được lựa chọn cục bộ không làm hỏng tương lai. Bài này là ví dụ nhập môn rất đẹp cho việc chứng minh đó.

## 3. Cách 1 — Thử mọi lựa chọn (tìm kiếm trạng thái)

Khi khách trả 20$ mà có cả hai cách thối, ta **chưa biết** cách nào tốt hơn → thử **cả hai**. Lưu tập các trạng thái `(five, ten)` có thể xảy ra sau mỗi khách; nếu tập rỗng thì thất bại.

Số trạng thái có thể lớn (`five`, `ten` tới `10^5` mỗi loại) ⇒ chậm, nhưng làm **oracle** rất tốt và giúp thấy rõ bản chất "lựa chọn".

In [5]:
def lemonade_all_choices(bills: list[int]) -> bool:
    states = {(0, 0)}                          # tập các (five, ten) có thể
    for bill in bills:
        new_states = set()
        for five, ten in states:
            if bill == 5:
                new_states.add((five + 1, ten))
            elif bill == 10:
                if five >= 1:
                    new_states.add((five - 1, ten + 1))
            else:  # 20
                if ten >= 1 and five >= 1:
                    new_states.add((five - 1, ten - 1))   # lựa chọn A: 10 + 5
                if five >= 3:
                    new_states.add((five - 3, ten))       # lựa chọn B: 5 + 5 + 5
        states = new_states
        if not states:
            return False
    return True

print(lemonade_all_choices([5, 5, 5, 10, 20]), lemonade_all_choices([5, 5, 10, 10, 20]))

True False


### Quan sát: tập trạng thái lớn dần

Mọi trạng thái trong tập có cùng tổng tiền (`5·five + 10·ten` như nhau), nên số trạng thái tối đa cỡ `O(n)`; nhưng mỗi khách phải duyệt cả tập ⇒ tổng thời gian có thể tới `O(n²)`. Hãy xem tập lớn dần khi có nhiều khách 20$ mà ta còn cả hai lựa chọn.

In [6]:
def count_states(bills):
    states = {(0, 0)}
    for bill in bills:
        nxt = set()
        for f, t in states:
            if bill == 5: nxt.add((f + 1, t))
            elif bill == 10 and f >= 1: nxt.add((f - 1, t + 1))
            elif bill == 20:
                if t >= 1 and f >= 1: nxt.add((f - 1, t - 1))
                if f >= 3: nxt.add((f - 3, t))
        states = nxt
    return len(states)

pattern = [5] * 40 + [10] * 15 + [20] * 10
print("số trạng thái cuối:", count_states(pattern), "với", len(pattern), "khách")

số trạng thái cuối: 8 với 65 khách


## 4. Insight — Tờ 5$ "quý" hơn tờ 10$

Khi khách trả 20$ và ta có cả `10 + 5` lẫn `5 + 5 + 5`:

- Tờ **5$** dùng được cho **mọi** loại thối (thối 10$ khách chỉ được bằng 5$; thối 20$ cũng cần ít nhất một tờ 5).
- Tờ **10$** chỉ dùng được cho một việc: thối khách 20$.

⇒ Nên **tiêu tờ 10 trước**, giữ lại tờ 5. Cụ thể: thối `10 + 5` nếu được, chỉ khi không có tờ 10 mới dùng `5 + 5 + 5`.

**So sánh hai trạng thái sau khi thối 15$:**
- Lựa chọn A (`10 + 5`): `(five - 1, ten - 1)`
- Lựa chọn B (`5+5+5`): `(five - 3, ten)`

A có **nhiều hơn 2 tờ 5** và **ít hơn 1 tờ 10**. Hai tờ 5 làm được mọi việc mà một tờ 10 làm được (thay `10 + 5` bằng `5 + 5 + 5`), và hơn thế. Nên trạng thái A **không bao giờ tệ hơn** B.

In [7]:
five, ten = 3, 1
A = (five - 1, ten - 1)
B_state = (five - 3, ten)
print("sau lựa chọn A (10+5):", A, "| sau lựa chọn B (5+5+5):", B_state)
print("Khách tiếp theo trả 10$ (cần 1 tờ 5) -> A thối được?", A[0] >= 1, "| B?", B_state[0] >= 1)

sau lựa chọn A (10+5): (2, 0) | sau lựa chọn B (5+5+5): (0, 1)
Khách tiếp theo trả 10$ (cần 1 tờ 5) -> A thối được? True | B? False


## 5. Lời giải chính — greedy hai biến đếm (từng đoạn rồi ghép)

**Đoạn 1:** `five = ten = 0`.

**Đoạn 2 — khách trả 5:** `five += 1`.

**Đoạn 3 — khách trả 10:** cần một tờ 5. Nếu `five == 0` → `False`; ngược lại `five -= 1; ten += 1`.

**Đoạn 4 — khách trả 20:** ưu tiên `10 + 5`:
```python
if ten > 0 and five > 0:   ten -= 1; five -= 1
elif five >= 3:            five -= 3
else:                      return False
```

**Đoạn 5:** qua hết hàng → `True`.

In [8]:
def lemonade_greedy(bills: list[int]) -> bool:
    five = ten = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0:
                return False
            five -= 1
            ten += 1
        else:  # bill == 20
            if ten > 0 and five > 0:
                ten -= 1
                five -= 1
            elif five >= 3:
                five -= 3
            else:
                return False
    return True

print(lemonade_greedy([5, 5, 5, 10, 20]), lemonade_greedy([5, 5, 10, 10, 20]))

True False


## 6. TRACE từng bước

### 6.1. Bảng trace ví dụ 2: `[5, 5, 10, 10, 20]`

| khách | trả | five trước | ten trước | hành động | five sau | ten sau |
|---|---|---|---|---|---|---|
| 0 | 5 | 0 | 0 | nhận 5 | 1 | 0 |
| 1 | 5 | 1 | 0 | nhận 5 | 2 | 0 |
| 2 | 10 | 2 | 0 | thối 5 | 1 | 1 |
| 3 | 10 | 1 | 1 | thối 5 | 0 | 2 |
| 4 | 20 | 0 | 2 | cần 15: không có 5 nào → **False** | | |

In [9]:
def trace_lemonade(bills: list[int]) -> bool:
    five = ten = 0
    print(f"bills={bills}")
    for i, bill in enumerate(bills):
        before = (five, ten)
        if bill == 5:
            five += 1; act = "nhận 5"
        elif bill == 10:
            if five == 0:
                print(f"khách {i}: trả 10, five={five} -> KHÔNG thối được -> False"); return False
            five -= 1; ten += 1; act = "thối 5"
        else:
            if ten > 0 and five > 0:
                ten -= 1; five -= 1; act = "thối 10+5"
            elif five >= 3:
                five -= 3; act = "thối 5+5+5"
            else:
                print(f"khách {i}: trả 20, (five,ten)={before} -> KHÔNG thối được -> False"); return False
        print(f"khách {i}: trả {bill:>2} | (five,ten) {before} -> {(five, ten)} | {act}")
    print("Phục vụ hết -> True")
    return True

trace_lemonade([5, 5, 10, 10, 20]);

bills=[5, 5, 10, 10, 20]
khách 0: trả  5 | (five,ten) (0, 0) -> (1, 0) | nhận 5
khách 1: trả  5 | (five,ten) (1, 0) -> (2, 0) | nhận 5
khách 2: trả 10 | (five,ten) (2, 0) -> (1, 1) | thối 5
khách 3: trả 10 | (five,ten) (1, 1) -> (0, 2) | thối 5
khách 4: trả 20, (five,ten)=(0, 2) -> KHÔNG thối được -> False


### 6.2. Trace ca mà greedy "cứu" được: `[5, 5, 5, 5, 10, 20, 10]`

Ở khách 20$, có cả hai lựa chọn. Greedy chọn `10 + 5`, giữ lại 2 tờ 5 → khách cuối trả 10 vẫn thối được. Nếu chọn `5 + 5 + 5` thì sẽ hết tờ 5 và **thất bại**.

In [10]:
trace_lemonade([5, 5, 5, 5, 10, 20, 10]);

bills=[5, 5, 5, 5, 10, 20, 10]
khách 0: trả  5 | (five,ten) (0, 0) -> (1, 0) | nhận 5
khách 1: trả  5 | (five,ten) (1, 0) -> (2, 0) | nhận 5
khách 2: trả  5 | (five,ten) (2, 0) -> (3, 0) | nhận 5
khách 3: trả  5 | (five,ten) (3, 0) -> (4, 0) | nhận 5
khách 4: trả 10 | (five,ten) (4, 0) -> (3, 1) | thối 5
khách 5: trả 20 | (five,ten) (3, 1) -> (2, 0) | thối 10+5
khách 6: trả 10 | (five,ten) (2, 0) -> (1, 1) | thối 5
Phục vụ hết -> True


In [11]:
def lemonade_bad_choice(bills):           # ưu tiên 5+5+5 (chiến lược tồi)
    five = ten = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0: return False
            five -= 1; ten += 1
        else:
            if five >= 3: five -= 3
            elif ten > 0 and five > 0: ten -= 1; five -= 1
            else: return False
    return True

print("ưu tiên 5+5+5:", lemonade_bad_choice([5, 5, 5, 5, 10, 20, 10]), "| greedy 10+5:", lemonade_greedy([5, 5, 5, 5, 10, 20, 10]))

ưu tiên 5+5+5: False | greedy 10+5: True


## 7. Code nộp LeetCode

In [12]:
class Solution:
    def lemonadeChange(self, bills: list[int]) -> bool:
        five = ten = 0
        for bill in bills:
            if bill == 5:
                five += 1
            elif bill == 10:
                if not five:
                    return False
                five -= 1
                ten += 1
            elif ten and five:
                ten -= 1
                five -= 1
            elif five >= 3:
                five -= 3
            else:
                return False
        return True

sol = Solution()
assert sol.lemonadeChange([5, 5, 5, 10, 20]) is True
assert sol.lemonadeChange([5, 5, 10, 10, 20]) is False
print("Solution PASS 2 ví dụ đề")

Solution PASS 2 ví dụ đề


`if not five:` nghĩa là `five == 0` (số 0 được coi là `False`). `elif ten and five:` nghĩa là cả hai khác 0.

## 8. Vì sao greedy đúng? Độ phức tạp

So sánh greedy với **một chiến lược bất kỳ** `OPT` (cùng hàng khách). Gọi `(f_g, t_g)` và `(f_o, t_o)` là số tờ 5, 10 của hai bên.

**Invariant:** khi cả hai còn phục vụ được, luôn có
1. `5·f_g + 10·t_g == 5·f_o + 10·t_o` (hai bên đã thu cùng số tiền và thối cùng số tiền, tờ 20 bỏ ra ngoài), tức `f_g + 2·t_g == f_o + 2·t_o`;
2. `f_g >= f_o` (greedy luôn có **ít nhất** bằng số tờ 5).

**Chứng minh theo từng khách (quy nạp):**
- Khách 5$: cả hai `f += 1` — giữ nguyên invariant.
- Khách 10$: `OPT` thối được ⇒ `f_o >= 1` ⇒ `f_g >= 1` ⇒ greedy cũng thối được; cả hai `f -= 1, t += 1`.
- Khách 20$, greedy có `t_g > 0, f_g > 0` ⇒ greedy dùng `10 + 5`: `f_g` giảm 1, còn `f_o` giảm 1 hoặc 3 ⇒ vẫn `f_g >= f_o`.
- Khách 20$, greedy có `t_g == 0` ⇒ `f_g = f_o + 2·t_o`. `OPT` thối được nghĩa là (`f_o >= 1` và `t_o >= 1`) hoặc `f_o >= 3`; cả hai trường hợp đều cho `f_g >= 3` ⇒ greedy dùng `5+5+5` được, và `f_g - 3 = f_o + 2·t_o - 3 >= ` số tờ 5 còn lại của `OPT`.
- Khách 20$, greedy có `t_g > 0` nhưng `f_g == 0` ⇒ `f_o == 0` ⇒ `OPT` cũng không thối được.

Vậy: nếu greedy thất bại ở khách nào thì mọi chiến lược khác cũng thất bại ở đó (hoặc sớm hơn). Greedy đúng.

**Khách 5$, 10$ không có lựa chọn** — mọi chiến lược làm giống nhau.

**Độ phức tạp:** `n = len(bills)`. Time `O(n)`, Space `O(1)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi lựa chọn (tập trạng thái) | `O(n²)` tệ nhất (tập tới `O(n)` trạng thái) | `O(n)` |
| 2. Greedy hai biến | `O(n)` | `O(1)` |
| 3. Greedy với `dict` ngăn kéo | `O(n)` | `O(1)` |

## 9. Cách 3 — Greedy tổng quát với "ngăn kéo" `dict`

Viết theo kiểu tổng quát: cần thối `change`, lần lượt dùng tờ **mệnh giá lớn nhất** có thể (10 rồi 5). Với hệ mệnh giá `5, 10` thì "lớn trước" là đúng (như đã chứng minh). Cách viết này dễ mở rộng sang biến thể.

In [13]:
def lemonade_drawer(bills: list[int]) -> bool:
    drawer = {5: 0, 10: 0}
    for bill in bills:
        change = bill - 5
        if bill in drawer:
            drawer[bill] += 1
        for d in (10, 5):                     # mệnh giá lớn trước
            while change >= d and drawer[d] > 0:
                change -= d
                drawer[d] -= 1
        if change != 0:
            return False
    return True

print(lemonade_drawer([5, 5, 5, 10, 20]), lemonade_drawer([5, 5, 10, 10, 20]), lemonade_drawer([5, 5, 5, 5, 10, 20, 10]))

True False True


Chú ý: nhận tờ tiền của khách **trước** khi thối là an toàn ở đây, vì tờ khách vừa đưa có mệnh giá **lớn hơn** số tiền cần thối, nên không bao giờ bị dùng để thối cho chính khách đó.

## 10. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Hai ví dụ đề.
2. Khách đầu tiên trả 10 hoặc 20 → `False` ngay.
3. Chỉ có khách trả 5.
4. Ca phân biệt greedy đúng/sai: `[5, 5, 5, 5, 10, 20, 10]`.
5. Thối 20 bằng `5 + 5 + 5` khi không có tờ 10: `[5, 5, 5, 20]`.

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.lemonadeChange([5, 5, 5, 10, 20]), True)
check("đề 2", sol.lemonadeChange([5, 5, 10, 10, 20]), False)
check("đầu trả 10", sol.lemonadeChange([10]), False)
check("đầu trả 20", sol.lemonadeChange([20, 5, 5]), False)
check("toàn 5", sol.lemonadeChange([5] * 7), True)
check("greedy quan trọng", sol.lemonadeChange([5, 5, 5, 5, 10, 20, 10]), True)
check("5+5+5", sol.lemonadeChange([5, 5, 5, 20]), True)
check("thiếu một tờ 5", sol.lemonadeChange([5, 5, 20]), False)
check("nhiều tờ 10", sol.lemonadeChange([5, 10] * 3 + [5, 5, 20]), True)
check("hết 5 vì nhiều tờ 10", sol.lemonadeChange([5, 10] * 3 + [20]), False)

PASS đề 1: True
PASS đề 2: False
PASS đầu trả 10: False
PASS đầu trả 20: False
PASS toàn 5: True
PASS greedy quan trọng: True
PASS 5+5+5: True
PASS thiếu một tờ 5: False
PASS nhiều tờ 10: True
PASS hết 5 vì nhiều tờ 10: False


### 10.1. Đối chiếu greedy với "thử mọi lựa chọn" (có seed)

In [15]:
import random
random.seed(860)
impls = [lemonade_greedy, lemonade_drawer, Solution().lemonadeChange]
true_count = 0
bad_choice_wrong = 0
for _ in range(3000):
    bills = random.choices([5, 5, 5, 10, 20], k=random.randint(1, 12))   # nhiều tờ 5 để có ca True
    expected = lemonade_all_choices(bills)
    true_count += expected
    for f in impls:
        assert f(bills) == expected, (bills, f)
    bad_choice_wrong += lemonade_bad_choice(bills) != expected
print(f"3000 hàng khách: greedy khớp oracle ({true_count} ca True)")
print(f"Chiến lược ưu tiên 5+5+5 sai ở {bad_choice_wrong} ca")

3000 hàng khách: greedy khớp oracle (1001 ca True)
Chiến lược ưu tiên 5+5+5 sai ở 15 ca


`random.choices(population, k=...)` chọn `k` phần tử **có lặp lại**; lặp số 5 trong `population` để tăng xác suất ra tờ 5.

### 10.2. Hiệu năng với `n = 10^5`

In [16]:
import time
big = [5] * 50000 + [10, 20] * 25000
t0 = time.perf_counter()
res = Solution().lemonadeChange(big)
print(res, f"{time.perf_counter() - t0:.4f}s")

True 0.0067s


## 11. Lỗi người mới hay gặp

### Lỗi 1: ưu tiên `5 + 5 + 5` khi thối 20

Đã thấy ở mục 6.2: tiêu tờ 5 quý giá làm khách 10$ sau đó không được thối.

In [17]:
print("sai :", lemonade_bad_choice([5, 5, 5, 5, 10, 20, 10]), "| đúng:", Solution().lemonadeChange([5, 5, 5, 5, 10, 20, 10]))

sai : False | đúng: True


### Lỗi 2: chỉ kiểm tra **tổng tiền** đủ

"Tổng tiền đang có ≥ tiền cần thối" là chưa đủ — hai tờ 10 có tổng 20 ≥ 15 nhưng không thối được 15.

In [18]:
def wrong_total(bills):
    total = 0
    for bill in bills:
        change = bill - 5
        if total < change:
            return False
        total += 5                 # nhận bill, trả change -> thực thu 5
    return True

print("sai :", wrong_total([5, 5, 10, 10, 20]), "| đúng:", Solution().lemonadeChange([5, 5, 10, 10, 20]))

sai : True | đúng: False


### Lỗi 3: viết `if ten > 0: ... elif five >= 3` nhưng quên kiểm tra `five > 0` trong nhánh `10 + 5`

Có tờ 10 nhưng **không** có tờ 5 vẫn không thối `10 + 5` được → `five` âm.

In [19]:
def wrong_missing_five(bills):
    five = ten = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0: return False
            five -= 1; ten += 1
        else:
            if ten > 0:                      # SAI: thiếu five > 0
                ten -= 1; five -= 1
            elif five >= 3:
                five -= 3
            else:
                return False
    return True

case = [5, 10, 20]
print("sai :", wrong_missing_five(case), "| đúng:", Solution().lemonadeChange(case))

sai : True | đúng: False


### Lỗi 4: đếm và "dùng" tờ 20 để thối

Tờ 20 không bao giờ dùng để thối (không có khoản thối nào ≥ 20). Đếm thêm biến `twenty` không sai nhưng thừa; dùng nó để thối thì sai.

## 12. Mẹo nhận dạng pattern

- Mô phỏng hàng đợi/giao dịch tuần tự với **tài nguyên đếm được** → giữ vài biến đếm trạng thái, xử lý từng phần tử một lần.
- Khi có **lựa chọn** tại mỗi bước, hỏi: "Lựa chọn nào giữ lại tài nguyên **linh hoạt** hơn?" → ứng viên cho greedy; sau đó **chứng minh** (exchange argument) hoặc kiểm tra bằng oracle thử-mọi-lựa-chọn.
- Greedy "mệnh giá lớn trước" đúng với hệ tiền **chuẩn** (như 5/10), nhưng **không** đúng với mọi hệ (xem biến thể 13.2).
- Bài liên quan: *Coin Change (322)* (khi greedy sai → DP), *Assign Cookies (455)*, *Gas Station (134)*.

## 13. Biến thể

### 13.1. Giá ly nước là `p`, mệnh giá tùy ý — mô phỏng với greedy "lớn trước"

In [20]:
def change_greedy(bills: list[int], price: int, denoms: list[int]) -> bool:
    drawer = {d: 0 for d in denoms}
    for bill in bills:
        drawer[bill] += 1
        change = bill - price
        for d in sorted(denoms, reverse=True):
            while change >= d and drawer[d] > 0:
                change -= d; drawer[d] -= 1
        if change:
            return False
    return True

print(change_greedy([5, 5, 5, 10, 20], 5, [5, 10, 20]))

True


`{d: 0 for d in denoms}` là **dict comprehension**; `sorted(..., reverse=True)` sắp xếp giảm dần.

### 13.2. Khi greedy "lớn trước" **sai**: mệnh giá `{1, 3, 4}`, cần thối 6

Greedy lấy `4` trước → còn `2` → `1 + 1` (3 tờ). Tối ưu là `3 + 3` (2 tờ). Nếu chỉ có tờ `4, 3, 3` trong ngăn kéo thì greedy lấy 4 rồi kẹt, trong khi `3 + 3` thành công.

In [21]:
def greedy_pay(amount, drawer):
    drawer = dict(drawer)
    for d in sorted(drawer, reverse=True):
        while amount >= d and drawer[d] > 0:
            amount -= d; drawer[d] -= 1
    return amount == 0

print("greedy thối 6 từ {4:1, 3:2}:", greedy_pay(6, {4: 1, 3: 2}), "| thực tế 3+3 được: True")

greedy thối 6 từ {4:1, 3:2}: False | thực tế 3+3 được: True


### 13.3. Đếm số khách tối đa phục vụ được (dừng khi hết khả năng)

In [22]:
def served_count(bills: list[int]) -> int:
    five = ten = 0
    for i, bill in enumerate(bills):
        if bill == 5:
            five += 1
        elif bill == 10:
            if not five: return i
            five -= 1; ten += 1
        elif ten and five:
            ten -= 1; five -= 1
        elif five >= 3:
            five -= 3
        else:
            return i
    return len(bills)

print(served_count([5, 5, 10, 10, 20]), served_count([5, 5, 5, 10, 20]))

4 5


## 14. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
Trace `[5, 5, 10, 20, 5, 5, 5, 5, 5, 5, 5, 5, 5, 10, 5, 5, 20, 5, 20, 5]`: sau mỗi khách 20$ còn bao nhiêu tờ 5, 10? Kết quả cuối?

### Bài 2 — Dự đoán output
Không chạy, đoán: `bool(0)`, `not 3`, `5 and 0`, `random.choices` có thể trả phần tử trùng không?

### Bài 3 — Điền TODO

In [23]:
def ex3_lemonade(bills: list[int]) -> bool:
    five = ten = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            # TODO
            pass
        else:
            # TODO: ưu tiên 10 + 5, sau đó 5 + 5 + 5
            pass
    return True

# assert ex3_lemonade([5, 5, 5, 10, 20]) is True
# assert ex3_lemonade([5, 5, 10, 10, 20]) is False

### Bài 4 — Trả về **số tờ 5 tối thiểu** cần có thêm ban đầu
Nếu được phép bắt đầu với `k` tờ 5, tìm `k` nhỏ nhất để phục vụ hết hàng (dùng greedy).

In [24]:
def ex4_min_initial_fives(bills: list[int]) -> int:
    # TODO
    pass

# assert ex4_min_initial_fives([5, 5, 10, 10, 20]) == 1
# assert ex4_min_initial_fives([20]) == 3
# assert ex4_min_initial_fives([5, 5, 5, 10, 20]) == 0

### Bài 5 — Biến thể giá 10$
Ly nước giá **10$**, khách trả `10`, `20` hoặc `50`. Hãy viết hàm kiểm tra thối được hết không (mệnh giá thối: 10, 20). Tờ 50 không dùng để thối.

In [25]:
def ex5_lemonade_10(bills: list[int]) -> bool:
    # TODO: thối 10 cho tờ 20; thối 40 cho tờ 50 (ưu tiên 20+20, rồi 20+10+10, rồi 10*4)
    pass

# assert ex5_lemonade_10([10, 10, 20, 50]) is False
# assert ex5_lemonade_10([10, 10, 10, 10, 50]) is True
# assert ex5_lemonade_10([10, 20, 10, 20, 50]) is True

### Bài 6 — Mini challenge: chứng minh bằng thực nghiệm
Viết hàm `oracle_ex5(bills)` thử **mọi** cách thối cho bài 5 (giống mục 3), rồi kiểm tra `ex5_lemonade_10` khớp oracle trên 1000 bộ ngẫu nhiên.

In [26]:
def ex6_oracle_10(bills: list[int]) -> bool:
    # TODO: tập trạng thái (ten, twenty)
    pass

## 15. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: dùng `trace_lemonade`... nhưng hãy tự tính trước.
- Bài 4: mỗi lần thiếu tờ 5, coi như "vay" thêm một tờ ban đầu.
- Bài 5: tờ 10 là "linh hoạt nhất" → tiêu tờ 20 trước khi thối 40.

**Mức 2 — cú pháp**
- Bài 3: nhánh 10: `if five == 0: return False`; nhánh 20: `if ten and five: ... elif five >= 3: ... else: return False`.
- Bài 4: chạy greedy; khi cần tờ 5 mà `five == 0` thì `k += 1` và coi như `five += 1`.

**Mức 3 — gần code**
- Bài 4: nhánh 20 không có tờ 10: thiếu `need = 3 - five` tờ nếu `five < 3` → `k += need`, `five += need`.
- Bài 6: `states = {(0, 0)}`; với 50: thử `(t, w-2)`, `(t-2, w-1)`, `(t-4, w)` nếu đủ.

## 16. Đáp án tham khảo

**Bài 2:** `False`, `False`, `0`, có (chọn có lặp lại).

**Bài 1:** xem trace ở cell ngay dưới — greedy phục vụ hết hàng, kết quả `True`; số tờ sau mỗi khách 20$ đọc trong cột `(five,ten)`.

In [27]:
print(bool(0), not 3, 5 and 0)
bills1 = [5, 5, 10, 20, 5, 5, 5, 5, 5, 5, 5, 5, 5, 10, 5, 5, 20, 5, 20, 5]
trace_lemonade(bills1);

False False 0
bills=[5, 5, 10, 20, 5, 5, 5, 5, 5, 5, 5, 5, 5, 10, 5, 5, 20, 5, 20, 5]
khách 0: trả  5 | (five,ten) (0, 0) -> (1, 0) | nhận 5
khách 1: trả  5 | (five,ten) (1, 0) -> (2, 0) | nhận 5
khách 2: trả 10 | (five,ten) (2, 0) -> (1, 1) | thối 5
khách 3: trả 20 | (five,ten) (1, 1) -> (0, 0) | thối 10+5
khách 4: trả  5 | (five,ten) (0, 0) -> (1, 0) | nhận 5
khách 5: trả  5 | (five,ten) (1, 0) -> (2, 0) | nhận 5
khách 6: trả  5 | (five,ten) (2, 0) -> (3, 0) | nhận 5
khách 7: trả  5 | (five,ten) (3, 0) -> (4, 0) | nhận 5
khách 8: trả  5 | (five,ten) (4, 0) -> (5, 0) | nhận 5
khách 9: trả  5 | (five,ten) (5, 0) -> (6, 0) | nhận 5
khách 10: trả  5 | (five,ten) (6, 0) -> (7, 0) | nhận 5
khách 11: trả  5 | (five,ten) (7, 0) -> (8, 0) | nhận 5
khách 12: trả  5 | (five,ten) (8, 0) -> (9, 0) | nhận 5
khách 13: trả 10 | (five,ten) (9, 0) -> (8, 1) | thối 5
khách 14: trả  5 | (five,ten) (8, 1) -> (9, 1) | nhận 5
khách 15: trả  5 | (five,ten) (9, 1) -> (10, 1) | nhận 5
khách 16: trả 20 | (five

In [28]:
def ans3_lemonade(bills):
    five = ten = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0:
                return False
            five -= 1; ten += 1
        else:
            if ten and five:
                ten -= 1; five -= 1
            elif five >= 3:
                five -= 3
            else:
                return False
    return True

def ans4_min_initial_fives(bills):
    five = ten = 0
    k = 0
    for bill in bills:
        if bill == 5:
            five += 1
        elif bill == 10:
            if five == 0:
                k += 1; five += 1
            five -= 1; ten += 1
        else:
            if ten:
                if five == 0:
                    k += 1; five += 1
                ten -= 1; five -= 1
            else:
                if five < 3:
                    k += 3 - five; five = 3
                five -= 3
    return k

def ans5_lemonade_10(bills):
    ten = twenty = 0
    for bill in bills:
        if bill == 10:
            ten += 1
        elif bill == 20:
            if not ten: return False
            ten -= 1; twenty += 1
        else:  # 50 -> thối 40
            if twenty >= 2: twenty -= 2
            elif twenty == 1 and ten >= 2: twenty -= 1; ten -= 2
            elif ten >= 4: ten -= 4
            else: return False
    return True

def ans6_oracle_10(bills):
    states = {(0, 0)}
    for bill in bills:
        nxt = set()
        for t, w in states:
            if bill == 10: nxt.add((t + 1, w))
            elif bill == 20:
                if t >= 1: nxt.add((t - 1, w + 1))
            else:
                if w >= 2: nxt.add((t, w - 2))
                if w >= 1 and t >= 2: nxt.add((t - 2, w - 1))
                if t >= 4: nxt.add((t - 4, w))
        states = nxt
        if not states: return False
    return True

assert ans3_lemonade([5, 5, 5, 10, 20]) is True and ans3_lemonade([5, 5, 10, 10, 20]) is False
assert ans4_min_initial_fives([5, 5, 10, 10, 20]) == 1
assert ans4_min_initial_fives([20]) == 3
assert ans4_min_initial_fives([5, 5, 5, 10, 20]) == 0
assert ans5_lemonade_10([10, 10, 20, 50]) is False
assert ans5_lemonade_10([10, 10, 10, 10, 50]) is True
assert ans5_lemonade_10([10, 20, 10, 20, 50]) is True
random.seed(8600)
for _ in range(1000):
    bs = random.choices([10, 10, 10, 20, 20, 50], k=random.randint(1, 10))
    assert ans5_lemonade_10(bs) == ans6_oracle_10(bs), bs
print("Đáp án bài 3-6 PASS (greedy bài 5 khớp oracle trên 1000 bộ)")

Đáp án bài 3-6 PASS (greedy bài 5 khớp oracle trên 1000 bộ)


## 17. Template nhớ nhanh
```python
five = ten = 0
for bill in bills:
    if bill == 5:   five += 1
    elif bill == 10:
        if not five: return False
        five -= 1; ten += 1
    elif ten and five: ten -= 1; five -= 1      # ưu tiên 10 + 5
    elif five >= 3:    five -= 3               # rồi mới 5 + 5 + 5
    else: return False
return True
```

**Câu thần chú:** *"Tờ 5 là vàng — tiêu tờ 10 trước."*

## 18. Checklist trước khi submit

- [ ] Nhánh 20$: thử `10 + 5` **trước** `5 + 5 + 5`?
- [ ] Nhánh `10 + 5` kiểm tra **cả** `ten > 0` **và** `five > 0`?
- [ ] Không chỉ kiểm tra tổng tiền?
- [ ] Không dùng tờ 20 để thối?
- [ ] Trả `False` **ngay** khi một khách không được thối, `True` sau vòng lặp?
- [ ] Đã test khách đầu trả 10/20, ca `[5, 5, 5, 5, 10, 20, 10]`?